# Topic 4: Streaming Joins

> **Phase 6 → Week 11 → Topic 4**

---

## Types of Streaming Joins

```
1. Stream-Static Join (stateless):
   stream.join(static_df, "key")
   ✓ No state required  ✓ Append output mode  ✓ Exactly-once
   The static side is loaded once. Use for enrichment (lookup tables).

2. Stream-Stream Join (stateful):
   stream1.join(stream2, "key")
   Stateful: Spark buffers rows from both sides waiting for a match.
   REQUIRES watermark on both streams to bound state.
   Without watermark: state grows forever → OOM.
   
   Use case: match payments to orders arriving in different topics.

Supported join types:
  Stream-Static:  inner, left outer, right outer, left semi, left anti
  Stream-Stream:  inner, left outer (requires watermark), right outer, full outer (Spark 3.3+)
```

---

## Stream-Stream Join State

```
orders stream:    O001 at 10:05, O002 at 10:07
payments stream:  P001 (for O001) at 10:08, P002 (for O002) at 10:15

Without watermark:
  Spark buffers O001 and O002 waiting for payment to arrive.
  Spark also buffers P001 waiting for an order to match.
  State NEVER evicted → OOM if orders never get payments.

With watermark on both streams:
  orders.withWatermark("order_time", "1 hour")
  payments.withWatermark("payment_time", "1 hour")
  .join(condition & event_time_constraint)

  Spark evicts order rows from state after 1 hour (no payment = dropped)
  Spark evicts payment rows from state after 1 hour (no order = dropped)
  State is bounded ✓
```

---

## Interview Cheat Sheet

**Q: What is a stream-stream join and what is the challenge?**
> A stream-stream join matches rows from two independent streams on a key. The challenge is that rows from the two streams may arrive at different times — an order arrives now but its matching payment arrives 10 minutes later. Spark must buffer rows from both sides in the State Store until a match is found or the row is too late (evicted by watermark). Without watermark, state grows forever. With watermark, state is bounded but late events that arrive after the watermark are dropped.

**Q: When do you use stream-static vs stream-stream join?**
> Stream-static: the right side is a fixed lookup table (product catalog, user data, tax rates) that changes infrequently. Load once, join each incoming event against it. Stream-stream: both sides are live streams with new data arriving continuously (orders + payments, clicks + impressions, login + logout events). Use stream-stream only when you genuinely have two streams to correlate.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time, os, shutil
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("Week11 - Streaming Joins") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

## Part 1: Stream-Static Join (Enrichment)

In [ ]:
# Static user profile lookup table
user_profiles = spark.createDataFrame([
    ("C1", "Alice",   "Mumbai",    "Gold"),
    ("C2", "Bob",     "Delhi",     "Silver"),
    ("C3", "Charlie", "Bangalore", "Bronze"),
    ("C4", "Diana",   "Chennai",   "Gold"),
    ("C5", "Eve",     "Hyderabad", "Silver"),
], ["customer_id", "name", "city", "tier"])

# Streaming orders
order_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("order_id",    F.concat(F.lit("O"), F.col("value").cast("string"))) \
    .withColumn("customer_id", F.concat(F.lit("C"), (F.col("value") % 5 + 1).cast("string"))) \
    .withColumn("amount",      (F.col("value") % 500 + 10).cast("double")) \
    .drop("value")

# Stream-static join: enrich orders with user profile
enriched = order_stream \
    .join(user_profiles, "customer_id", "left") \
    .select("order_id", "customer_id", "name", "city", "tier", "amount")

q = enriched.writeStream \
    .format("memory") \
    .queryName("enriched_stream") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(10)
print("Enriched orders (stream joined with static user profiles):")
spark.sql("""
    SELECT tier, city, count(*) as orders, round(avg(amount),0) as avg_amount
    FROM enriched_stream
    GROUP BY tier, city ORDER BY tier, city
""").show()

q.stop()

## Part 2: Stream-Stream Join (Orders ↔ Payments)

In [ ]:
# Stream-stream join: match orders with payments
# Simulated using two file streams so we control timing

ORDERS_DIR   = "/tmp/ss_orders"
PAYMENTS_DIR = "/tmp/ss_payments"
CKPT_SS     = "/tmp/ckpt_ss_join"

for p in [ORDERS_DIR, PAYMENTS_DIR, CKPT_SS]:
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p)

now = datetime.now()

def ts(minutes_ago=0):
    return (now - timedelta(minutes=minutes_ago)).strftime("%Y-%m-%d %H:%M:%S")

order_schema = StructType([
    StructField("order_id",    StringType()),
    StructField("customer_id", StringType()),
    StructField("amount",      DoubleType()),
    StructField("order_time",  StringType()),
])
payment_schema = StructType([
    StructField("order_id",     StringType()),
    StructField("payment_id",   StringType()),
    StructField("payment_time", StringType()),
    StructField("method",       StringType()),
])

# Write orders
spark.createDataFrame([
    ("O001", "C1", 100.0, ts(2)),
    ("O002", "C2", 250.0, ts(1)),
    ("O003", "C1", 500.0, ts(0)),
], ["order_id", "customer_id", "amount", "order_time"]) \
    .coalesce(1).write.json(f"{ORDERS_DIR}/batch1")

# Write payments (some matching, some not yet)
spark.createDataFrame([
    ("O001", "P101", ts(1), "UPI"),
    ("O002", "P102", ts(0), "Card"),
    # O003 has no payment yet → will be handled by outer join
], ["order_id", "payment_id", "payment_time", "method"]) \
    .coalesce(1).write.json(f"{PAYMENTS_DIR}/batch1")

# Build streaming DataFrames
orders_stream = spark.readStream.schema(order_schema).json(ORDERS_DIR) \
    .withColumn("order_time", F.to_timestamp("order_time")) \
    .withWatermark("order_time", "5 minutes")

payments_stream = spark.readStream.schema(payment_schema).json(PAYMENTS_DIR) \
    .withColumn("payment_time", F.to_timestamp("payment_time")) \
    .withWatermark("payment_time", "5 minutes")

# Stream-stream inner join on order_id with time constraint
# The time constraint bounds how old the buffered state can be
joined = orders_stream.join(
    payments_stream,
    (
        orders_stream.order_id == payments_stream.order_id
    ) & (
        # Payment must arrive within 10 minutes of order
        payments_stream.payment_time.between(
            orders_stream.order_time,
            orders_stream.order_time + F.expr("INTERVAL 10 MINUTES")
        )
    ),
    how="inner"
).select(
    orders_stream.order_id,
    "customer_id", "amount",
    "payment_id", "method",
    orders_stream.order_time,
    payments_stream.payment_time,
)

q_ss = joined.writeStream \
    .format("memory") \
    .queryName("matched_orders") \
    .outputMode("append") \
    .option("checkpointLocation", CKPT_SS) \
    .trigger(once=True) \
    .start()

q_ss.awaitTermination()
print("Stream-stream join results (orders matched with payments):")
spark.sql("""
    SELECT order_id, customer_id, amount, payment_id, method
    FROM matched_orders
    ORDER BY order_id
""").show()
print("Note: O003 has no payment → not in inner join result")

## Part 3: Join Type Reference

In [ ]:
print("""
Streaming Join Reference:
════════════════════════════════════════════════════════════════

Stream-Static Join:
  stream.join(static_df, "key", "left")   ← left, inner, right all work
  No state required. Append output mode always works.
  Static side loaded at query start (re-load: restart query or use foreachBatch).

Stream-Stream Inner Join:
  Both sides watermarked.
  Optional: time range constraint to bound state.
  
  stream1.withWatermark("ts1", "1 hour")
  stream2.withWatermark("ts2", "1 hour")
  .join(
      stream1.key == stream2.key
      & stream2.ts2.between(stream1.ts1 - expr("INTERVAL 10 MINUTES"),
                             stream1.ts1 + expr("INTERVAL 10 MINUTES"))
  )

  outputMode: append (result emitted only when window is closed)

Stream-Stream Left Outer Join:
  Requires watermark on BOTH streams.
  Left-side rows with no match are emitted with null right-side columns
  AFTER the watermark passes them (not immediately!).
  Use for: "orders that have no matching payment"

  stream1.withWatermark("ts1", "1 hour")
  .join(stream2.withWatermark("ts2", "1 hour"), condition, "left")
  outputMode: append

State size for stream-stream join:
  Proportional to: (rows per second) × (watermark duration)
  If processing 1000 orders/sec with 1-hour watermark:
    state ≈ 1000 × 3600 = 3.6M order rows buffered at any time
  → Choose the smallest watermark that covers your late-arrival window
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Create a stream-static join where the static side is a DataFrame loaded from a Parquet file. What happens when the Parquet file is updated while the streaming query is running? How would you make the join always use the latest version?
2. Implement a stream-stream join between a `clicks` stream and an `impressions` stream on `ad_id`. Use a 30-minute watermark. What is the maximum state size in rows if you process 500 clicks/sec and 2000 impressions/sec?
3. You do a left outer join between orders and payments with a 30-minute watermark. An order O999 arrives at 10:00 and its payment arrives at 10:45. Is the payment included in the result? What row appears in the output for O999?
4. What is the difference between a stream-static join and a stream-stream join in terms of output mode support? Explain why they differ.
5. You need to match login events with logout events for the same `user_id` to compute session duration. Write the stream-stream join structure (no need to run it — just the join condition and watermark setup).